# Results Analysis

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
import re
import warnings
warnings.filterwarnings('ignore')

# Paths
RESULTS_DIR = Path('results')
EXPORT_DIR  = Path('analysis')
FIGS_DIR    = Path('analysis/figures')
EXPORT_DIR.mkdir(exist_ok=True)
FIGS_DIR.mkdir(parents=True, exist_ok=True)

# Bootstrap
N_BOOT = 2000
SEED   = 42

# Experiment config
DATASETS = ['adult', 'compas', 'german']

DATASET_PROTECTED = {
    'adult':  {'sex': 1, 'race': 1},
    'compas': {'sex': 1, 'race': 1},
    'german': {'sex': 1, 'age': 1},
}

# Condition parsing — order matters: longest / most specific keys first.
# These are the EXACT internal tags carried into the paper as-is.
FNAME_TO_COND = [
    ('D4r_high_0',     'D4r_0'),    # not used in current filenames (kept for safety)
    ('D4r_0',          'D4r_0'),
    ('D4r_D1',         'D4r_D1'),
    ('D4r_D2',         'D4r_D2'),
    ('D4r_high',       'D4r_0'),    # legacy fallback for files that lacked the demo suffix
    ('D4a',            'D4a'),
    ('D4b_D1',         'D4b_D1'),
    ('D4b_D2',         'D4b_D2'),
    ('ZS_D5',          'ZS_D5'),
    ('ZS_D1',          'ZS_D1'),
    ('D5_decontam',    'D5_decontam'),
    ('D3_resampled',   'D3'),
    ('D2_fair_causal', 'D2'),
    ('D1_original',    'D1'),
]

# Conditions that are evaluated on BOTH test=D1 and test=D2 (dual-test).
# The rest are test=D1 only.
D4_DUAL_TEST_CONDITIONS = {'D4a', 'D4b_D1', 'D4b_D2', 'D4r_0', 'D4r_D1', 'D4r_D2'}

HF_MODELS = [
    'meta-llama-Llama-3-1-8B-Instruct',
    'Qwen-Qwen2-5-7B-Instruct',
    'Qwen-Qwen2-5-14B-Instruct',
    'openai-gpt-oss-20b',
    'google-gemma-4-E4B-it',
    'google-gemma-4-31B-it',
]
GROQ_MODELS = [
    'llama-3-3-70b-versatile',
    'qwen-qwen3-32b',
]
MODELS      = HF_MODELS + GROQ_MODELS
D4R_MODELS  = { 'qwen-qwen3-32b',
                'google-gemma-4-E4B-it', 'google-gemma-4-31B-it'}
KNOWN_MODELS = sorted(MODELS, key=len, reverse=True)

# Short display names for plots
MODEL_SHORT = {
    'meta-llama-Llama-3-1-8B-Instruct':      'Llama-3.1-8B',
    'Qwen-Qwen2-5-7B-Instruct':              'Qwen2.5-7B',
    'Qwen-Qwen2-5-14B-Instruct':             'Qwen2.5-14B',
    'openai-gpt-oss-20b':                    'GPT-OSS-20B',
    'google-gemma-4-E4B-it':                 'Gemma-4-E4B',
    'google-gemma-4-31B-it':                 'Gemma-4-31B',
    'llama-3-3-70b-versatile':               'Llama-3.3-70B',
    'qwen-qwen3-32b':                        'Qwen3-32B',
}

# Internal-tag ordering used in figures and tables (paper uses these directly).
COND_ORDER = [
    'D1', 'D2', 'D3',
    'D4a', 'D4b_D1', 'D4b_D2',
    'D4r_0', 'D4r_D1', 'D4r_D2',
    'ZS_D1', 'ZS_D5', 'D5_decontam',
]

# Test-set labels shown in the paper (the column / panel header).
TEST_SET_LABEL = {'D1': 'Test=Original', 'D2': 'Test=FairCausal'}

# Plot style
plt.rcParams.update({
    'figure.dpi': 150,
    'font.size': 9,
    'axes.titlesize': 10,
    'axes.labelsize': 9,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 8,
    'savefig.bbox': 'tight',
    'savefig.dpi': 300,
})

print(f'Results dir: {RESULTS_DIR.resolve()}')
print(f'Exists: {RESULTS_DIR.exists()}')
print(f'Models: {len(MODELS)}')
print(f'Conditions: {len(COND_ORDER)}  (dual-test on: {sorted(D4_DUAL_TEST_CONDITIONS)})')


## 2. Load results

In [ ]:
_TESTSET_RE = re.compile(r'_test(D1|D2)(?:_|$)')


def parse_filename(fpath: Path) -> dict:
    """Extract (condition, model, dataset, test_set) from a result-file path.

    test_set is derived from the `_testD1`/`_testD2` suffix in the stem when
    present (only D4* files), otherwise defaults to 'D1' (the canonical test
    set for data-level / ZS / D5 conditions).
    """
    stem = fpath.stem

    # Test-set tag
    m = _TESTSET_RE.search(stem)
    test_set = m.group(1) if m else 'D1'

    # Condition
    condition = 'unknown'
    for key, cond in FNAME_TO_COND:
        if key in stem:
            condition = cond
            break

    # Model tag
    model_tag = 'unknown'
    for tag in KNOWN_MODELS:
        if tag in stem:
            model_tag = tag
            break
    if model_tag == 'unknown':
        try:
            after_llm = stem.split('_LLM_', 1)[1]
            raw = after_llm.rsplit('_table_', 1)[0]
            # Strip the _testD1/_testD2 suffix from raw if present
            raw = _TESTSET_RE.sub('', raw)
            for prefix in ['FewShot_', 'ZSCoT_D4a_', 'FSCoT_D4b_', 'ZeroShot_', 'D4r_high_']:
                if raw.startswith(prefix):
                    raw = raw[len(prefix):]
                    break
            model_tag = raw
        except Exception:
            pass

    parent      = fpath.parent.name
    grandparent = fpath.parent.parent.name
    dataset = grandparent if grandparent in DATASETS else parent
    return {'condition': condition, 'model': model_tag,
            'dataset': dataset, 'test_set': test_set}


def load_tables(results_dir: Path):
    """Load performance and fairness tables, attaching parsed metadata.

    Each row gets condition/model/dataset/test_set columns. If a CSV already
    has a 'test_set' column (written by the new pipeline) it is preserved;
    otherwise we use the value parsed from the filename.
    """
    perf_files = sorted(results_dir.rglob('*table_performance.csv'))
    fair_files = sorted(results_dir.rglob('*table_fairness.csv'))
    print(f'{len(perf_files)} performance files | {len(fair_files)} fairness files')

    def _load(files):
        dfs = []
        for f in files:
            meta = parse_filename(f)
            df = pd.read_csv(f)
            for k, v in meta.items():
                if k == 'test_set' and 'test_set' in df.columns:
                    # Trust the column written by the pipeline; backfill NaNs.
                    df['test_set'] = df['test_set'].fillna(v)
                else:
                    df[k] = v
            dfs.append(df)
        return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

    return _load(perf_files), _load(fair_files)


df_perf, df_fair = load_tables(RESULTS_DIR)

# Sanity checks
print('Conditions:', sorted(df_fair['condition'].unique()))
print('Models:    ', sorted(df_fair['model'].unique()))
print('Datasets:  ', sorted(df_fair['dataset'].unique()))
print('Test sets: ', sorted(df_fair['test_set'].unique()))

# Audit: any 'unknown' parsing
unk = df_fair[(df_fair['condition'] == 'unknown') | (df_fair['model'] == 'unknown')]
if not unk.empty:
    print(f'WARNING: {len(unk)} rows with unknown condition or model.')

# D4* dual-test coverage
d4_rows = df_fair[df_fair['condition'].isin(D4_DUAL_TEST_CONDITIONS)]
covg = d4_rows.groupby(['condition', 'test_set']).size().unstack(fill_value=0)
print('\nD4* test-set coverage (rows per condition x test_set):')
print(covg)


## 3. Bootstrap confidence intervals

In [ ]:
# Metric functions
def _tpr(yt, yp): return np.sum((yp==1)&(yt==1)) / max(np.sum(yt==1), 1)
def _fpr(yt, yp): return np.sum((yp==1)&(yt==0)) / max(np.sum(yt==0), 1)
def _ppp(yp, m):  return np.sum(yp[m]==1) / max(np.sum(m), 1)

def eod(yt, yp, g, p):
    return _tpr(yt[g!=p], yp[g!=p]) - _tpr(yt[g==p], yp[g==p])
def di(yt, yp, g, p):
    d = _ppp(yp, g==p)
    return _ppp(yp, g!=p) / d if d > 0 else np.nan
def spd(yt, yp, g, p):
    return _ppp(yp, g!=p) - _ppp(yp, g==p)
def od(yt, yp, g, p):
    return (_fpr(yt[g!=p], yp[g!=p]) - _fpr(yt[g==p], yp[g==p])) + \
           (_tpr(yt[g!=p], yp[g!=p]) - _tpr(yt[g==p], yp[g==p]))

METRIC_FNS = {'EOD': eod, 'DI': di, 'SPD': spd, 'OD': od}


def bootstrap_ci(y_true, y_pred, group, priv, n_boot=N_BOOT, seed=SEED):
    rng  = np.random.default_rng(seed)
    n    = len(y_true)
    boot = {m: [] for m in METRIC_FNS}
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        for m, fn in METRIC_FNS.items():
            boot[m].append(fn(y_true[idx], y_pred[idx], group[idx], priv))
    out = {}
    for m, samples in boot.items():
        valid = np.array([s for s in samples if not np.isnan(s)])
        out[m] = (round(float(np.percentile(valid, 2.5)), 4),
                  round(float(np.percentile(valid, 97.5)), 4)) if len(valid) else (np.nan, np.nan)
    return out


# Compute over all prediction files. Each file is one (condition, model, dataset, test_set)
# bucket. If the file carries a `test_set` column it overrides the filename-derived value.
pred_files = sorted(RESULTS_DIR.rglob('*_predictions.csv'))
print(f'{len(pred_files)} prediction files found')

boot_rows = []
for fpath in pred_files:
    meta = parse_filename(fpath)
    if 'unknown' in (meta['condition'], meta['model']):
        print(f'[!] Skipping (unparsed): {fpath.name}')
        continue
    df = pd.read_csv(fpath)
    if 'y_true' not in df.columns or 'y_pred' not in df.columns:
        continue
    # Trust the column when present; otherwise the filename tag.
    test_set = df['test_set'].iloc[0] if 'test_set' in df.columns and not df['test_set'].isna().all() else meta['test_set']

    yt = df['y_true'].values
    yp = df['y_pred'].values
    for feature, priv in DATASET_PROTECTED.get(meta['dataset'], {}).items():
        if feature not in df.columns:
            continue
        g   = df[feature].values
        cis = bootstrap_ci(yt, yp, g, priv)
        for metric, fn in METRIC_FNS.items():
            pt = round(float(fn(yt, yp, g, priv)), 4)
            boot_rows.append({
                'condition': meta['condition'], 'model': meta['model'],
                'dataset': meta['dataset'], 'test_set': test_set,
                'feature': feature, 'metric': metric,
                'point_est': pt, 'ci_lo': cis[metric][0], 'ci_hi': cis[metric][1],
                'n': len(yt),
            })

df_boot = pd.DataFrame(boot_rows)
print(f'Bootstrap done: {len(df_boot)} rows')
print('Test sets in df_boot:', sorted(df_boot['test_set'].unique()) if not df_boot.empty else 'empty')


## 4. Accuracy summary

In [ ]:
if not df_perf.empty and 'A' in df_perf.columns:
    df_acc = (
        df_perf.groupby(['model', 'condition', 'dataset', 'test_set'])['A']
        .mean().reset_index()
        .rename(columns={'A': 'Accuracy'})
    )
    df_acc['model_short'] = df_acc['model'].map(MODEL_SHORT).fillna(df_acc['model'])

    # Two views:
    #  (a) Canonical test=D1 view — all conditions appear here.
    #  (b) D4* dual-test view — show D4* on both test sets side by side.
    pivot_acc_d1 = (df_acc[df_acc['test_set'] == 'D1']
                    .pivot_table(index=['model_short', 'condition'],
                                 columns='dataset', values='Accuracy').round(3))
    print('ACCURACY by model x condition x dataset (test=D1, all conditions)')
    print(pivot_acc_d1.to_string())

    d4_acc = df_acc[df_acc['condition'].isin(D4_DUAL_TEST_CONDITIONS)].copy()
    if not d4_acc.empty:
        pivot_acc_d4 = d4_acc.pivot_table(
            index=['model_short', 'condition'],
            columns=['dataset', 'test_set'], values='Accuracy'
        ).round(3)
        print('\nACCURACY for D4* across both test sets (Test=Original=D1, Test=FairCausal=D2)')
        print(pivot_acc_d4.to_string())


## 5. EOD heatmap: model × condition (one per dataset)

In [ ]:
if not df_boot.empty:
    # ---- View 1: full condition x model heatmap (test=D1 only) ----
    df_eod_d1 = (
        df_boot[(df_boot['metric'] == 'EOD') & (df_boot['test_set'] == 'D1')]
        .groupby(['dataset', 'model', 'condition'])['point_est']
        .mean().reset_index()
    )
    df_eod_d1['model_short'] = df_eod_d1['model'].map(MODEL_SHORT).fillna(df_eod_d1['model'])
    model_order = [MODEL_SHORT.get(m, m) for m in MODELS
                   if MODEL_SHORT.get(m, m) in df_eod_d1['model_short'].values]
    cond_order  = [c for c in COND_ORDER if c in df_eod_d1['condition'].values]

    fig, axes = plt.subplots(1, 3, figsize=(17, 5), sharey=False)
    vabs = df_eod_d1['point_est'].abs().quantile(0.95) if not df_eod_d1.empty else 0.2

    for ax, dataset in zip(axes, DATASETS):
        sub = df_eod_d1[df_eod_d1['dataset'] == dataset]
        pivot = sub.pivot_table(index='model_short', columns='condition', values='point_est')
        pivot = pivot.reindex(index=model_order, columns=cond_order)
        sns.heatmap(
            pivot, ax=ax, cmap='RdYlGn_r', center=0,
            vmin=-vabs, vmax=vabs,
            annot=True, fmt='.2f', annot_kws={'size': 7},
            linewidths=0.4, linecolor='white',
            cbar_kws={'label': 'EOD', 'shrink': 0.8},
        )
        ax.set_title(dataset.capitalize(), fontweight='bold')
        ax.set_xlabel('Condition')
        ax.set_ylabel('Model' if ax == axes[0] else '')
        ax.tick_params(axis='x', rotation=45)
        ax.tick_params(axis='y', rotation=0)

    fig.suptitle('EOD by model and condition (test=Original=D1)\n'
                 'Green = more fair (EOD near 0), Red = less fair', y=1.02)
    plt.tight_layout()
    plt.savefig(FIGS_DIR / 'eod_heatmap_testD1.pdf')
    plt.savefig(FIGS_DIR / 'eod_heatmap_testD1.png')
    plt.show()
    print('eod_heatmap_testD1 saved')


    # ---- View 2: D4* only — Test=D1 | Test=D2 side-by-side per dataset ----
    df_eod_d4 = (
        df_boot[(df_boot['metric'] == 'EOD') &
                (df_boot['condition'].isin(D4_DUAL_TEST_CONDITIONS))]
        .groupby(['dataset', 'model', 'condition', 'test_set'])['point_est']
        .mean().reset_index()
    )
    if not df_eod_d4.empty:
        df_eod_d4['model_short'] = df_eod_d4['model'].map(MODEL_SHORT).fillna(df_eod_d4['model'])
        d4_cond_order = [c for c in COND_ORDER if c in D4_DUAL_TEST_CONDITIONS]
        vabs_d4 = df_eod_d4['point_est'].abs().quantile(0.95)

        fig, axes = plt.subplots(len(DATASETS), 2, figsize=(13, 3.4 * len(DATASETS)), sharex=True)
        for row_idx, dataset in enumerate(DATASETS):
            for col_idx, ts in enumerate(['D1', 'D2']):
                ax = axes[row_idx, col_idx]
                sub = df_eod_d4[(df_eod_d4['dataset'] == dataset) &
                                (df_eod_d4['test_set'] == ts)]
                pivot = sub.pivot_table(index='model_short', columns='condition', values='point_est')
                pivot = pivot.reindex(index=model_order, columns=d4_cond_order)
                sns.heatmap(
                    pivot, ax=ax, cmap='RdYlGn_r', center=0,
                    vmin=-vabs_d4, vmax=vabs_d4,
                    annot=True, fmt='.2f', annot_kws={'size': 7},
                    linewidths=0.4, linecolor='white',
                    cbar=(col_idx == 1),
                    cbar_kws={'label': 'EOD', 'shrink': 0.8} if col_idx == 1 else None,
                )
                title = f'{dataset.capitalize()} — {TEST_SET_LABEL[ts]}'
                ax.set_title(title, fontweight='bold')
                ax.set_xlabel('Condition' if row_idx == len(DATASETS) - 1 else '')
                ax.set_ylabel('Model' if col_idx == 0 else '')
                ax.tick_params(axis='x', rotation=45)
                ax.tick_params(axis='y', rotation=0)

        fig.suptitle('EOD for D4* conditions: Test=Original (left) vs Test=FairCausal (right)\n'
                     'Mirrors the prior FLAI mitigated-model evaluation on Original/Fair Data', y=1.005)
        plt.tight_layout()
        plt.savefig(FIGS_DIR / 'eod_heatmap_d4_dualtest.pdf')
        plt.savefig(FIGS_DIR / 'eod_heatmap_d4_dualtest.png')
        plt.show()
        print('eod_heatmap_d4_dualtest saved')


## 6. Effect of data strategy: D1 → D2 → D3

In [ ]:
if not df_boot.empty:
    df_d123 = df_boot[
        (df_boot['metric'] == 'EOD') &
        (df_boot['condition'].isin(['D1', 'D2', 'D3'])) &
        (df_boot['test_set'] == 'D1')
    ].copy()
    df_d123 = df_d123.groupby(['dataset', 'model', 'condition'])['point_est'].mean().reset_index()
    df_d123['model_short'] = df_d123['model'].map(MODEL_SHORT).fillna(df_d123['model'])
    df_d123['eod_abs'] = df_d123['point_est'].abs()

    fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=False)
    colors = {m: c for m, c in zip(
        [MODEL_SHORT.get(m, m) for m in MODELS],
        plt.cm.tab20.colors
    )}

    for ax, dataset in zip(axes, DATASETS):
        sub = df_d123[df_d123['dataset'] == dataset]
        for model_short, grp in sub.groupby('model_short'):
            grp = grp.set_index('condition').reindex(['D1', 'D2', 'D3'])
            ax.plot([0, 1, 2], grp['eod_abs'].values,
                    marker='o', linewidth=1.4, markersize=5,
                    color=colors.get(model_short, 'gray'),
                    label=model_short, alpha=0.85)
        ax.set_xticks([0, 1, 2])
        ax.set_xticklabels(['D1\n(original)', 'D2\n(fair-causal)', 'D3\n(resampled)'])
        ax.set_title(dataset.capitalize(), fontweight='bold')
        ax.set_ylabel('|EOD|' if ax == axes[0] else '')
        ax.set_ylim(bottom=0)
        ax.axhline(0, color='black', linewidth=0.5, linestyle='--')
        ax.grid(axis='y', alpha=0.3)

    handles = [mpatches.Patch(color=colors.get(m, 'gray'), label=m)
               for m in [MODEL_SHORT.get(m, m) for m in MODELS] if m in colors]
    fig.legend(handles=handles, loc='lower center', ncol=4,
               bbox_to_anchor=(0.5, -0.12), frameon=False)
    fig.suptitle('Effect of data strategy on fairness (|EOD|, test=Original)\n'
                 'Lower is better. D2=fair-causal examples, D3=resampled', y=1.02)
    plt.tight_layout()
    plt.savefig(FIGS_DIR / 'data_strategy.pdf', bbox_inches='tight')
    plt.savefig(FIGS_DIR / 'data_strategy.png', bbox_inches='tight')
    plt.show()
    print('data_strategy saved')


## 7. CoT effect: accuracy vs EOD scatter

In [ ]:
if not df_boot.empty and not df_perf.empty and 'A' in df_perf.columns:
    # Conditions considered as "interventions" for the scatter
    cot_conds = ['D4a', 'D4b_D1', 'D4b_D2', 'D4r_0', 'D4r_D1', 'D4r_D2']
    # D1 (ICL-Original, test=D1) is the baseline reference
    baseline_cond = 'D1'

    df_cot_eod = (
        df_boot[(df_boot['metric'] == 'EOD') &
                (df_boot['condition'].isin(cot_conds + [baseline_cond]))]
        .groupby(['dataset', 'model', 'condition', 'test_set'])['point_est']
        .mean().reset_index()
        .rename(columns={'point_est': 'EOD'})
    )
    df_cot_acc = (
        df_perf[df_perf['condition'].isin(cot_conds + [baseline_cond])]
        .groupby(['dataset', 'model', 'condition', 'test_set'])['A']
        .mean().reset_index()
        .rename(columns={'A': 'Accuracy'})
    )
    df_cot = df_cot_eod.merge(df_cot_acc, on=['dataset', 'model', 'condition', 'test_set'])
    df_cot['model_short'] = df_cot['model'].map(MODEL_SHORT).fillna(df_cot['model'])
    df_cot['eod_abs'] = df_cot['EOD'].abs()

    cond_markers = {'D1': 'o', 'D4a': 's', 'D4b_D1': '^', 'D4b_D2': 'v',
                    'D4r_0': 'P', 'D4r_D1': 'D', 'D4r_D2': 'X'}
    cond_colors  = {'D1': '#888888',
                    'D4a': '#DD8452',
                    'D4b_D1': '#55A868', 'D4b_D2': '#2E7D32',
                    'D4r_0': '#C44E52', 'D4r_D1': '#9C27B0', 'D4r_D2': '#6A1B9A'}

    # One scatter per (test_set, dataset). D1 baseline always plotted with its test=D1 value.
    for ts in ['D1', 'D2']:
        fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=False)
        for ax, dataset in zip(axes, DATASETS):
            sub_ts = df_cot[(df_cot['dataset'] == dataset) & (df_cot['test_set'] == ts)]
            sub_base = df_cot[(df_cot['dataset'] == dataset) &
                              (df_cot['condition'] == baseline_cond) &
                              (df_cot['test_set'] == 'D1')]
            # Plot baseline
            ax.scatter(sub_base['Accuracy'], sub_base['eod_abs'],
                       marker=cond_markers[baseline_cond], color=cond_colors[baseline_cond],
                       s=60, alpha=0.85, label=baseline_cond, zorder=3)
            d1_by_model = sub_base.set_index('model_short')
            # Plot intervention conditions
            for cond in cot_conds:
                s = sub_ts[sub_ts['condition'] == cond]
                if s.empty:
                    continue
                ax.scatter(s['Accuracy'], s['eod_abs'],
                           marker=cond_markers[cond], color=cond_colors[cond],
                           s=60, alpha=0.85, label=cond, zorder=3)
                # Connect D1→intervention lines per model
                for _, row in s.iterrows():
                    ms = row['model_short']
                    if ms in d1_by_model.index:
                        ax.plot([d1_by_model.loc[ms, 'Accuracy'], row['Accuracy']],
                                [d1_by_model.loc[ms, 'eod_abs'], row['eod_abs']],
                                color=cond_colors[cond], alpha=0.2, linewidth=0.7)
            ax.set_xlabel('Accuracy')
            ax.set_ylabel('|EOD|' if ax == axes[0] else '')
            ax.set_title(dataset.capitalize(), fontweight='bold')
            ax.grid(alpha=0.3)

        handles = [mpatches.Patch(color=cond_colors[c], label=c) for c in [baseline_cond] + cot_conds]
        fig.legend(handles=handles, loc='lower center', ncol=7,
                   bbox_to_anchor=(0.5, -0.08), frameon=False,
                   title=f'Conditions evaluated on {TEST_SET_LABEL[ts]} '
                         f'(baseline D1 always Test=Original)')
        fig.suptitle(f'Accuracy vs |EOD| trade-off — {TEST_SET_LABEL[ts]}\n'
                     f'Bottom-right = best (high accuracy, low |EOD|)', y=1.02)
        plt.tight_layout()
        plt.savefig(FIGS_DIR / f'cot_tradeoff_test{ts}.pdf', bbox_inches='tight')
        plt.savefig(FIGS_DIR / f'cot_tradeoff_test{ts}.png', bbox_inches='tight')
        plt.show()
        print(f'cot_tradeoff_test{ts} saved')


## 8. Model prior: ZS_D1 vs D1 EOD comparison

In [ ]:
if not df_boot.empty:
    df_zs = (
        df_boot[(df_boot['metric'] == 'EOD') &
                (df_boot['condition'].isin(['D1', 'ZS_D1'])) &
                (df_boot['test_set'] == 'D1')]
        .groupby(['dataset', 'model', 'condition'])['point_est'].mean().reset_index()
    )
    df_zs['model_short'] = df_zs['model'].map(MODEL_SHORT).fillna(df_zs['model'])
    df_zs_pivot = df_zs.pivot_table(
        index=['dataset', 'model_short'], columns='condition', values='point_est'
    ).reset_index()
    df_zs_pivot = df_zs_pivot.dropna(subset=['D1', 'ZS_D1'])
    df_zs_pivot['delta'] = df_zs_pivot['ZS_D1'] - df_zs_pivot['D1']  # positive = ZS is worse

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for ax, dataset in zip(axes, DATASETS):
        sub = df_zs_pivot[df_zs_pivot['dataset'] == dataset].sort_values('delta')
        colors_bar = ['#d73027' if v > 0 else '#1a9850' for v in sub['delta']]
        ax.barh(sub['model_short'], sub['delta'], color=colors_bar, edgecolor='white', height=0.6)
        ax.axvline(0, color='black', linewidth=0.8)
        ax.set_title(dataset.capitalize(), fontweight='bold')
        ax.set_xlabel('ΔEOD (ZS_D1 − D1)\nPositive = zero-shot is less fair')
        ax.grid(axis='x', alpha=0.3)
        if ax != axes[0]:
            ax.set_yticklabels([])

    fig.suptitle('Model prior effect: zero-shot vs few-shot (ΔEOD = ZS_D1 − D1, test=Original)\n'
                 'Red = zero-shot increases unfairness, Green = zero-shot improves fairness', y=1.02)
    plt.tight_layout()
    plt.savefig(FIGS_DIR / 'prior_effect.pdf', bbox_inches='tight')
    plt.savefig(FIGS_DIR / 'prior_effect.png', bbox_inches='tight')
    plt.show()
    print('prior_effect saved')


## 9. Cross-dataset consistency heatmap

In [ ]:
if not df_boot.empty:
    # Rank models by |EOD| within each dataset for D1 baseline (test=D1)
    df_rank = (
        df_boot[(df_boot['metric'] == 'EOD') &
                (df_boot['condition'] == 'D1') &
                (df_boot['test_set'] == 'D1')]
        .groupby(['dataset', 'model'])['point_est']
        .apply(lambda x: x.abs().mean()).reset_index()
        .rename(columns={'point_est': 'eod_abs'})
    )
    df_rank['rank'] = df_rank.groupby('dataset')['eod_abs'].rank(method='min')
    df_rank['model_short'] = df_rank['model'].map(MODEL_SHORT).fillna(df_rank['model'])

    pivot_rank = df_rank.pivot_table(
        index='model_short', columns='dataset', values='rank'
    ).reindex(index=[MODEL_SHORT.get(m, m) for m in MODELS])

    fig, ax = plt.subplots(figsize=(6, 6))
    n_models = len(pivot_rank)
    sns.heatmap(
        pivot_rank, ax=ax,
        cmap='RdYlGn_r',
        vmin=1, vmax=n_models,
        annot=True, fmt='.0f', annot_kws={'size': 9},
        linewidths=0.5, linecolor='white',
        cbar_kws={'label': 'Rank (1 = most fair)', 'shrink': 0.7},
    )
    ax.set_title('Cross-dataset fairness ranking (D1 baseline, |EOD|, test=Original)\n'
                 'Rank 1 = lowest |EOD| (most fair)', fontweight='bold')
    ax.set_xlabel('Dataset')
    ax.set_ylabel('Model')
    ax.tick_params(axis='x', rotation=0)
    ax.tick_params(axis='y', rotation=0)
    plt.tight_layout()
    plt.savefig(FIGS_DIR / 'cross_dataset_rank.pdf', bbox_inches='tight')
    plt.savefig(FIGS_DIR / 'cross_dataset_rank.png', bbox_inches='tight')
    plt.show()
    print('cross_dataset_rank saved')


## 9b. D4* test-set robustness: Δ|EOD| between Test=Original and Test=FairCausal

For every D4* condition, plot the difference in |EOD| between the two test sets.
Mirrors the prior FLAI work's claim that a properly mitigated model maintains fairness
*independently of the test data*. A value close to 0 means the intervention's
fairness behaviour generalises across test distributions.

In [ ]:
if not df_boot.empty:
    df_d4 = (
        df_boot[(df_boot['metric'] == 'EOD') &
                (df_boot['condition'].isin(D4_DUAL_TEST_CONDITIONS))]
        .groupby(['dataset', 'model', 'condition', 'test_set'])['point_est']
        .mean().reset_index()
    )
    df_d4['eod_abs'] = df_d4['point_est'].abs()
    df_d4['model_short'] = df_d4['model'].map(MODEL_SHORT).fillna(df_d4['model'])

    # Pivot to get Test=D1 and Test=D2 columns side by side
    pivot = df_d4.pivot_table(
        index=['dataset', 'model_short', 'condition'],
        columns='test_set', values='eod_abs'
    ).reset_index().dropna(subset=['D1', 'D2'])
    pivot['delta_abs'] = pivot['D2'] - pivot['D1']  # positive = Test=Fair gives MORE |EOD|

    d4_cond_order = [c for c in COND_ORDER if c in D4_DUAL_TEST_CONDITIONS]

    fig, axes = plt.subplots(1, 3, figsize=(16, 5.5), sharey=False)
    for ax, dataset in zip(axes, DATASETS):
        sub = pivot[pivot['dataset'] == dataset]
        pv = sub.pivot_table(index='model_short', columns='condition', values='delta_abs')
        pv = pv.reindex(index=[MODEL_SHORT.get(m, m) for m in MODELS],
                        columns=d4_cond_order)
        vabs = max(abs(pv.values[~np.isnan(pv.values)]).max(), 0.05) if pv.notna().any().any() else 0.1
        sns.heatmap(
            pv, ax=ax, cmap='RdBu_r', center=0,
            vmin=-vabs, vmax=vabs,
            annot=True, fmt='.2f', annot_kws={'size': 7},
            linewidths=0.4, linecolor='white',
            cbar_kws={'label': 'Δ|EOD|  (Test=FairCausal − Test=Original)', 'shrink': 0.8},
        )
        ax.set_title(dataset.capitalize(), fontweight='bold')
        ax.set_xlabel('Condition')
        ax.set_ylabel('Model' if ax == axes[0] else '')
        ax.tick_params(axis='x', rotation=45)
        ax.tick_params(axis='y', rotation=0)

    fig.suptitle('Test-set robustness of D4* interventions: Δ|EOD| (FairCausal − Original)\n'
                 'Blue = mitigation generalises (less |EOD| on FairCausal); '
                 'Red = mitigation drifts (more |EOD| on FairCausal); '
                 'White ≈ robust', y=1.04)
    plt.tight_layout()
    plt.savefig(FIGS_DIR / 'd4_testset_robustness.pdf', bbox_inches='tight')
    plt.savefig(FIGS_DIR / 'd4_testset_robustness.png', bbox_inches='tight')
    plt.show()
    print('d4_testset_robustness saved')


## 10. Export CSV + LaTeX tables

In [ ]:
def df_to_latex(df, caption, label):
    return df.to_latex(
        index=True, escape=True, na_rep='—',
        float_format='{:.3f}'.format,
        caption=caption, label=label,
        column_format='l' * (df.index.nlevels + len(df.columns)),
        multirow=True,
    )

# Accuracy summary — split into test=D1 (all conds) and D4* dual-test
if not df_perf.empty and 'A' in df_perf.columns:
    df_acc.to_csv(EXPORT_DIR / 'accuracy_summary.csv', index=False)
    (EXPORT_DIR / 'table_accuracy_testD1.tex').write_text(
        df_to_latex(pivot_acc_d1,
                    'Accuracy by model, condition and dataset (test=Original).',
                    'tab:accuracy_testD1')
    )
    if 'pivot_acc_d4' in dir():
        (EXPORT_DIR / 'table_accuracy_d4_dualtest.tex').write_text(
            df_to_latex(pivot_acc_d4,
                'Accuracy for D4* conditions on both test sets '
                '(Test=Original=D1, Test=FairCausal=D2).',
                'tab:accuracy_d4_dualtest')
        )
    print('accuracy tables saved')

# Bootstrap CI — full CSV with test_set column
if not df_boot.empty:
    df_boot.to_csv(EXPORT_DIR / 'bootstrap_ci.csv', index=False)
    print('bootstrap_ci.csv')

    # Per-metric LaTeX tables, one per test_set bucket.
    # We emit two flavours:
    #   table_<metric>_ci_testD1.tex : all conditions, test=D1
    #   table_<metric>_ci_d4_dualtest.tex : only D4* conditions, dataset x test_set columns
    for metric in ['EOD', 'SPD', 'DI', 'OD']:
        df_m = df_boot[df_boot['metric'] == metric].copy()
        if df_m.empty:
            continue
        df_m['value'] = df_m.apply(
            lambda r: f"{r['point_est']:.3f} [{r['ci_lo']:.3f}, {r['ci_hi']:.3f}]", axis=1
        )
        df_m['model_short'] = df_m['model'].map(MODEL_SHORT).fillna(df_m['model'])

        # Flavour 1: test=D1 only
        df_m_d1 = df_m[df_m['test_set'] == 'D1']
        pivot1 = df_m_d1.pivot_table(
            index=['model_short', 'condition'],
            columns=['dataset', 'feature'],
            values='value', aggfunc='first'
        )
        pivot1.index.names = ['Model', 'Condition']
        latex1 = pivot1.to_latex(
            index=True, escape=True, na_rep='—',
            caption=f"{metric} with 95\\% bootstrap CIs (test=Original, $B={N_BOOT}$ resamples).",
            label=f"tab:{metric.lower()}_ci_testD1",
            column_format='l' * (pivot1.index.nlevels + len(pivot1.columns)),
            multirow=True,
        )
        (EXPORT_DIR / f'table_{metric.lower()}_ci_testD1.tex').write_text(latex1)
        print(f'[✓] table_{metric.lower()}_ci_testD1.tex')

        # Flavour 2: D4* across both test sets
        df_m_d4 = df_m[df_m['condition'].isin(D4_DUAL_TEST_CONDITIONS)]
        if not df_m_d4.empty:
            pivot2 = df_m_d4.pivot_table(
                index=['model_short', 'condition'],
                columns=['dataset', 'test_set', 'feature'],
                values='value', aggfunc='first'
            )
            pivot2.index.names = ['Model', 'Condition']
            latex2 = pivot2.to_latex(
                index=True, escape=True, na_rep='—',
                caption=f"{metric} for D4* conditions across both test sets "
                        f"(95\\% bootstrap CIs, $B={N_BOOT}$ resamples).",
                label=f"tab:{metric.lower()}_ci_d4_dualtest",
                column_format='l' * (pivot2.index.nlevels + len(pivot2.columns)),
                multirow=True,
            )
            (EXPORT_DIR / f'table_{metric.lower()}_ci_d4_dualtest.tex').write_text(latex2)
            print(f'[✓] table_{metric.lower()}_ci_d4_dualtest.tex')

print('\nAll exports done:', EXPORT_DIR.resolve())


## 11. Package tar.gz

In [ ]:
import tarfile, datetime

timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
tar_name  = f'results_package_{timestamp}.tar.gz'

with tarfile.open(tar_name, 'w:gz') as tar:
    for d in [RESULTS_DIR, EXPORT_DIR]:
        if d.exists():
            n = sum(1 for _ in d.rglob('*') if _.is_file())
            tar.add(d, arcname=d.name)
            print(f'[+] {d.name}/  ({n} files)')
        else:
            print(f'[!] {d} not found, skipping')

size_mb = Path(tar_name).stat().st_size / 1e6
print(f'\n{tar_name}  ({size_mb:.1f} MB)')